In [ ]:
import torch
import torch.nn.functional as F
import random


block_size = 5      
embedding_dim = 10  
hidden_layer = 300  
batch_size = 64     
learning_rate = 0.1 
iterations = 50000  

words = open('names.txt', 'r').read().splitlines()
chars = sorted(list(set(''.join(words))))
stoi = {s: i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}
vocab_size = len(itos)

X, Y = [], []
for w in words:
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        context = context[1:] + [ix]
X = torch.tensor(X)
Y = torch.tensor(Y)

g = torch.Generator().manual_seed(2147483647)
C  = torch.randn((vocab_size, embedding_dim),            generator=g, requires_grad=True)
W1 = torch.randn((block_size * embedding_dim, hidden_layer), generator=g, requires_grad=True)
b1 = torch.randn(hidden_layer,                            generator=g, requires_grad=True)
W2 = torch.randn((hidden_layer, vocab_size),              generator=g, requires_grad=True)
b2 = torch.randn(vocab_size,                             generator=g, requires_grad=True)

parameters = [C, W1, b1, W2, b2]

print(f"Training on {len(X)} samples...")
for i in range(iterations):
    
    ix = torch.randint(0, X.shape[0], (batch_size,), generator=g)
    Xb, Yb = X[ix], Y[ix]
    
    emb = C[Xb] 
    h = torch.tanh(emb.view(-1, block_size * embedding_dim) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Yb)
    
    for p in parameters:
        p.grad = None
    loss.backward()
    
    lr = learning_rate if i < (iterations * 0.8) else learning_rate / 10
    for p in parameters:
        p.data += -lr * p.grad

    if i % 5000 == 0:
        print(f"Step {i:7d}/{iterations}: loss {loss.item():.4f}")

with torch.no_grad():
    emb = C[X]
    h = torch.tanh(emb.view(-1, block_size * embedding_dim) @ W1 + b1)
    logits = h @ W2 + b2
    total_loss = F.cross_entropy(logits, Y)
    print(f"\n---> FINAL TOTAL LOSS: {total_loss.item():.4f}")

Training on 228146 samples...
Step       0/50000: loss 32.6139
Step    5000/50000: loss 2.6475
Step   10000/50000: loss 2.8438
Step   15000/50000: loss 2.3400
Step   20000/50000: loss 2.3543
Step   25000/50000: loss 2.6229
Step   30000/50000: loss 2.5459
Step   35000/50000: loss 2.3696
Step   40000/50000: loss 2.2998
Step   45000/50000: loss 2.1488

---> FINAL TOTAL LOSS: 2.2442
